In [33]:
from demo_flowsheet_structured import FS
from idaes.core.util.units_of_measurement import report_quantity
from idaes.core.util.model_statistics import number_activated_blocks, number_activated_constraints, number_variables, degrees_of_freedom
from idaes.core.base.unit_model import UnitModelBlockData
from pyomo.network import Arc

In [5]:
FS.run_steps()

get indexes of first step '-' and last step '-' in steps ['build', 'set_solver', 'initialize', 'set_operating_conditions', 'set_scaling', 'solve_initial', 'set_autoscaling', 'add_costing', 'initialize_costing', 'setup_optimization', 'solve_optimization']


2026-05-14 02:57:35 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.M01.inlet_1_state[0.0].eq_comp[benzene]
2026-05-14 02:57:35 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.M01.inlet_1_state[0.0].eq_comp[toluene]
2026-05-14 02:57:35 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.M01.inlet_1_state[0.0].eq_phase_equilibrium[benzene]
2026-05-14 02:57:35 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.M01.inlet_1_state[0.0].eq_phase_equilibrium[toluene]
2026-05-14 02:57:35 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.M01.inlet_1_state[0.0].eq_P_vap[benzene]
2026-05-14 02:57:35 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.M01.inlet_1_state[0.0].eq_P_vap[toluene]
2026-05-14 02:57:35 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.M01.inlet_2_state[0.0].eq_comp[benzene]
2026-05-14 02:57:35 [WARNING] idaes.core.util.scaling: Missing scaling factor for fs.M01.inlet_2_

In [15]:
f03.report??

Signature: f03.report(time_point=0, dof=False, ostream=None, prefix='')
Docstring: <no docstring>
Source:   
    def report(self, time_point=0, dof=False, ostream=None, prefix=""):

        time_point = float(time_point)

        if ostream is None:
            ostream = sys.stdout

        # Get DoF and model stats
        if dof:
            dof_stat = degrees_of_freedom(self)
            nv = number_variables(self)
            nc = number_activated_constraints(self)
            nb = number_activated_blocks(self)

        # Get components to report in performance section
        performance = self._get_performance_contents(time_point=time_point)

        # Get stream table
        stream_table = self._get_stream_table_contents(time_point=time_point)

        # Set model type output
        if hasattr(self, "is_flowsheet") and self.is_flowsheet:
            model_type = "Flowsheet"
        else:
            model_type = "Unit"

        # Write output
        max_str_length = 84
      

In [ ]:


def report(model, time_point=0, dof=True, prefix=""):

        time_point = float(time_point)

        # Get DoF and model stats
        if dof:
            dof_stat = degrees_of_freedom(model)
            nv = number_variables(model)
            nc = number_activated_constraints(model)
            nb = number_activated_blocks(model)

        # Get components to report in performance section
        performance = model._get_performance_contents(time_point=time_point)
        if performance is None:
             performance = {}
        else:
            for section in ("vars",):
                for k, v in performance.get(section, {}).items():
                    if hasattr(v, "value"):
                        performance[section][k] = {"value": report_quantity(v).m, "units": str(report_quantity(v).u), "fixed": v.fixed, "bounds": v.bounds}

        # Get stream table
        try:
            stream_table = model._get_stream_table_contents(time_point=time_point)
            stream_dict = stream_table.to_dict()
            stream_dict["Units"] = {k: str(v) for k, v in stream_dict["Units"].items()}
        except AttributeError:
             stream_dict = {}

        # Set model type output
        if hasattr(model, "is_flowsheet") and model.is_flowsheet:
            model_type = "Flowsheet"
        else:
            model_type = "Unit"

        return dict(performance=performance, stream_table=stream_dict, model_type=model_type, dof={"dof": dof_stat, "num_var": nv, "num_act_constraints": nc, "num_act_blocks": nb})

In [41]:
def has_report(c):
    return isinstance(c, UnitModelBlockData)  and hasattr(c, "_get_performance_contents")

In [42]:
unit_reports = {}
for obj in FS.model.fs.component_objects(Arc):
    streams = list(obj.values()) if obj.is_indexed() else [obj]
    for s in streams:
        for node in (s.source.parent_block(), s.dest.parent_block()):
            if not node in unit_reports and has_report(node):
                print(node.name)
                unit_reports[node] = report(node)

fs.M01
fs.H02
fs.F03


In [43]:
unit_reports

{<idaes.core.base.process_block._ScalarMixer at 0x73a3faf55860>: {'performance': {},
  'stream_table': {'Units': {'flow_mol': 'mole / second',
    'mole_frac_comp benzene': 'dimensionless',
    'mole_frac_comp toluene': 'dimensionless',
    'temperature': 'kelvin',
    'pressure': 'pascal'},
   'inlet_1': {'flow_mol': 1.0,
    'mole_frac_comp benzene': 0.99999,
    'mole_frac_comp toluene': 1e-05,
    'temperature': 370.0,
    'pressure': 101325.0},
   'inlet_2': {'flow_mol': 1.0,
    'mole_frac_comp benzene': 1e-05,
    'mole_frac_comp toluene': 0.99999,
    'temperature': 380.0,
    'pressure': 130000.0},
   'Outlet': {'flow_mol': 2.0,
    'mole_frac_comp benzene': 0.5,
    'mole_frac_comp toluene': 0.5,
    'temperature': 368.1219881350423,
    'pressure': 101324.99999999999}},
  'model_type': 'Unit',
  'dof': {'dof': 0,
   'num_var': 71,
   'num_act_constraints': 61,
   'num_act_blocks': 4}},
 <idaes.core.base.process_block._ScalarHeater at 0x73a3faf9e120>: {'performance': {'vars':

In [59]:
isinstance(FS.model.fs.F03, UnitModelBlockData)

True

In [35]:
type(f03)

idaes.core.base.process_block._ScalarFlash